# Control vs. TOU Shifted Residential Load

14-line plot showing the daily average hourly load profile for each of the 7 days in the dataset.

- **Gray lines** — control group daily averages
- **Orange lines** — TOU shifted group daily averages
- Each line is the mean kWh per hour across all households for that day
- Shaded band: peak pricing window

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
PEAK_START  = 17            # peak window start hour (inclusive, 1-24 scale)
PEAK_END    = 19            # peak window end hour (inclusive)
OUTPUT_FILE = 'tou_vs_control_load_profile.png'

In [ ]:
import io
import random
import zipfile

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.colab import files

## 1 · Upload data files

In [ ]:
print('Upload control_profile.csv and tou_shifted_profile.zip')
uploaded = files.upload()

## 2 · Load data

In [ ]:
# Control group
ctrl = pd.read_csv(io.BytesIO(uploaded['control_profile.csv']))

# TOU shifted group — unzip then read whichever CSV is inside
with zipfile.ZipFile(io.BytesIO(uploaded['tou_shifted_profile.zip'])) as z:
    csv_name = next(n for n in z.namelist() if n.endswith('.csv'))
    print(f'Found inside zip: {csv_name}')
    tou = pd.read_csv(z.open(csv_name))

print(f'Control rows : {len(ctrl):,}   |   columns: {list(ctrl.columns)}')
print(f'TOU rows     : {len(tou):,}   |   columns: {list(tou.columns)}')

## 3 · Prepare data

In [ ]:
HOURS = [str(h) for h in range(1, 25)]   # column names '1' … '24'
x     = np.arange(1, 25)                 # numeric x positions

# ── Per-day averages across all households ────────────────────────────────────
ctrl_daily = ctrl.groupby('date')[HOURS].mean()
tou_daily  = tou.groupby('date')[HOURS].mean()

dates = sorted(set(ctrl_daily.index) | set(tou_daily.index))
print(f'{len(dates)} days found: {dates}')

## 4 · Plot

In [ ]:
CTRL_COLOR = '#555555'
TOU_COLOR  = '#E87722'

fig, ax = plt.subplots(figsize=(14, 7))

# ── 7 control lines (gray gradient) + 7 TOU lines (orange gradient) ──────────
n_days = len(dates)
ctrl_alphas = np.linspace(0.35, 1.0, n_days)
tou_alphas  = np.linspace(0.35, 1.0, n_days)

for i, date in enumerate(dates):
    label_c = f'Control {date}'   if i == 0 else None
    label_t = f'TOU {date}'       if i == 0 else None

    if date in ctrl_daily.index:
        ax.plot(x, ctrl_daily.loc[date].values.astype(float),
                color=CTRL_COLOR, alpha=ctrl_alphas[i],
                linewidth=1.8, zorder=2, label=label_c)

    if date in tou_daily.index:
        ax.plot(x, tou_daily.loc[date].values.astype(float),
                color=TOU_COLOR, alpha=tou_alphas[i],
                linewidth=1.8, zorder=2, label=label_t)

# ── Peak window shading ───────────────────────────────────────────────────────
ax.axvspan(PEAK_START - 0.5, PEAK_END + 0.5,
           alpha=0.12, color='#FFD700', zorder=0)

blend    = ax.get_xaxis_transform()
peak_mid = (PEAK_START + PEAK_END) / 2

def _hour_to_ampm(h):
    h0 = h - 1
    if h0 == 0:   return '12am'
    if h0 < 12:   return f'{h0}am'
    if h0 == 12:  return '12pm'
    return f'{h0 - 12}pm'

peak_label = f'Peak Window\n({_hour_to_ampm(PEAK_START)}–{_hour_to_ampm(PEAK_END + 1)})'
ax.text(peak_mid, 0.97, peak_label,
        transform=blend, ha='center', va='top',
        fontsize=9, color='#7a6a00',
        bbox=dict(boxstyle='round,pad=0.3', fc='#fffde0', ec='none', alpha=0.7))

# ── Axes formatting ───────────────────────────────────────────────────────────
tick_hours = range(1, 25, 2)
ax.set_xticks(list(tick_hours))
ax.set_xticklabels([_hour_to_ampm(h) for h in tick_hours], fontsize=9)
ax.set_xlim(0.5, 24.5)

ax.set_xlabel('Hour of Day', fontsize=11, labelpad=8)
ax.set_ylabel('kWh', fontsize=11, labelpad=8)
ax.set_title('Control vs. TOU Shifted Residential Load — Daily Averages',
             fontsize=14, fontweight='bold', pad=12)

ax.grid(axis='y', linestyle='--', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# ── Legend: one entry per group per day ──────────────────────────────────────
ctrl_handles = [
    plt.Line2D([0], [0], color=CTRL_COLOR, alpha=ctrl_alphas[i],
               linewidth=1.8, label=f'Control  {dates[i]}')
    for i in range(n_days)
]
tou_handles = [
    plt.Line2D([0], [0], color=TOU_COLOR, alpha=tou_alphas[i],
               linewidth=1.8, label=f'TOU      {dates[i]}')
    for i in range(n_days)
]
peak_patch = mpatches.Patch(color='#FFD700', alpha=0.4, label='Peak Window')

ax.legend(handles=ctrl_handles + tou_handles + [peak_patch],
          fontsize=8, framealpha=0.9,
          loc='upper left', borderpad=0.8, ncol=2)

plt.tight_layout()
fig.savefig(OUTPUT_FILE, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {OUTPUT_FILE}')